In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\dmane\Downloads\Cogniv AI")
WHATSAPP_AUDIO = PROJECT_ROOT / "WhatsApp Audio 2026-09-12 at 10.11.50.mpeg"
MICROPHONE_WAV_CANDIDATES = [
    PROJECT_ROOT / "mic_test.wav",
    PROJECT_ROOT / "mic_clean.wav",
    PROJECT_ROOT / "live_utterance.wav",
]

print("Project root:", PROJECT_ROOT)
print("WhatsApp benchmark file:", WHATSAPP_AUDIO)
print("WhatsApp file exists:", WHATSAPP_AUDIO.exists())
print("Local microphone WAV candidates:")
for candidate in MICROPHONE_WAV_CANDIDATES:
    print(f"  {candidate.name}: {candidate.exists()}")

Project root: C:\Users\dmane\Downloads\Cogniv AI
WhatsApp benchmark file: C:\Users\dmane\Downloads\Cogniv AI\WhatsApp Audio 2026-09-12 at 10.11.50.mpeg
WhatsApp file exists: True
Local microphone WAV candidates:
  mic_test.wav: True
  mic_clean.wav: True
  live_utterance.wav: True


In [2]:
import importlib.metadata
import sys

import torch


def installed_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


print("=" * 60)
print("ENVIRONMENT AND GPU STATUS")
print("=" * 60)
print("Python executable:", sys.executable)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers version:", installed_version("transformers"))
print("Torchaudio version:", installed_version("torchaudio"))

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(properties.total_memory / (1024 ** 3), 2))
else:
    print("GPU: unavailable; inference will use CPU")

print("=" * 60)

ENVIRONMENT AND GPU STATUS
Python executable: c:\Users\dmane\anaconda3\python.exe
PyTorch version: 2.14.0+cu130
CUDA available: True
Transformers version: 5.17.0
Torchaudio version: 2.11.0+cu130
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM (GB): 6.0


In [3]:
import importlib.util
import subprocess
import sys

required_packages = {
    "transformers": "transformers",
    "torchaudio": "torchaudio",
}
missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Missing packages:", ", ".join(missing_packages))
    print("Installing only the missing packages...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *missing_packages,
    ])
else:
    print("Dependency check passed.")
    print("transformers and torchaudio are already installed; no installation needed.")

print("Required dependencies are ready. Restart the kernel only if pip installed a package into a different active kernel.")

Dependency check passed.
transformers and torchaudio are already installed; no installation needed.
Required dependencies are ready. Restart the kernel only if pip installed a package into a different active kernel.


In [4]:
import torch

MODEL_ID = "ai4bharat/indic-conformer-600m-multilingual"
TARGET_SAMPLE_RATE = 16_000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SUPPORTED_LANGUAGE_CODES = {
    "as", "bn", "brx", "doi", "gu", "hi", "kn", "kok", "ks", "mai",
    "ml", "mni", "mr", "ne", "or", "pa", "sa", "sat", "sd", "ta", "te", "ur",
}

print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("Expected sample rate:", TARGET_SAMPLE_RATE, "Hz")
print("Published language codes:", ", ".join(sorted(SUPPORTED_LANGUAGE_CODES)))
print("English (en) is not listed as supported by this model.")

Model: ai4bharat/indic-conformer-600m-multilingual
Device: cuda
Expected sample rate: 16000 Hz
Published language codes: as, bn, brx, doi, gu, hi, kn, kok, ks, mai, ml, mni, mr, ne, or, pa, sa, sat, sd, ta, te, ur
English (en) is not listed as supported by this model.


In [6]:
import os
import time

import torch
from dotenv import load_dotenv
from transformers import AutoModel


env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    load_dotenv(env_file, override=False)

hf_token = os.getenv("HF_TOKEN")

print("=" * 60)
print("LOADING INDICCONFORMER ASR MODEL")
print("=" * 60)
print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("Hugging Face token available:", bool(hf_token))
print("Loading with the official Transformers remote-code interface...")

model_load_started = time.perf_counter()
model_kwargs = {
    "trust_remote_code": True,
}
if hf_token:
    model_kwargs["token"] = hf_token

model = AutoModel.from_pretrained(
    MODEL_ID,
    **model_kwargs,
)
model = model.to(DEVICE)
model.eval()
model_load_time = time.perf_counter() - model_load_started

print(f"Model loaded in {model_load_time:.2f} seconds.")
print("Model is ready for CTC and RNNT decoding.")

LOADING INDICCONFORMER ASR MODEL
Model: ai4bharat/indic-conformer-600m-multilingual
Device: cuda
Hugging Face token available: True
Loading with the official Transformers remote-code interface...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual.
403 Client Error. (Request ID: Root=1-6aa55801-4da2f456131b746c4969cfb7;d8fc0e3e-987c-44de-bd7c-10fdcd8e597a)

Cannot access gated repo for url https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual/resolve/main/config.json.
Access to model ai4bharat/indic-conformer-600m-multilingual is restricted and you are not in the authorized list. Visit https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual to ask for access.

In [ ]:
from pathlib import Path

import torch
import torchaudio


def preprocess_audio(audio_path):
    """Load supported audio, convert it to mono 16 kHz float32, and normalize safely."""
    audio_path = Path(audio_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    waveform, source_sample_rate = torchaudio.load(str(audio_path))
    if waveform.ndim != 2:
        raise ValueError(f"Expected audio shaped [channels, samples], got {tuple(waveform.shape)}")

    waveform = waveform.to(dtype=torch.float32)
    waveform = waveform.mean(dim=0, keepdim=True)

    if source_sample_rate != TARGET_SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(
            orig_freq=source_sample_rate,
            new_freq=TARGET_SAMPLE_RATE,
        )
        waveform = resampler(waveform)

    peak = waveform.abs().max()
    if peak > 0:
        waveform = waveform / peak

    return waveform.contiguous()


def _gpu_memory_gb():
    if DEVICE.type != "cuda":
        return 0.0, 0.0
    allocated = torch.cuda.memory_allocated(DEVICE) / (1024 ** 3)
    peak = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3)
    return allocated, peak


def _run_inference(audio_path, language, decoder):
    if language == "auto":
        raise ValueError(
            "IndicConformer requires an explicit language code. "
            "Automatic language detection is not provided by this model interface."
        )
    if language not in SUPPORTED_LANGUAGE_CODES:
        raise ValueError(
            f"Unsupported language code {language!r}. "
            f"Use one of: {', '.join(sorted(SUPPORTED_LANGUAGE_CODES))}."
        )
    if decoder not in {"ctc", "rnnt"}:
        raise ValueError("decoder must be 'ctc' or 'rnnt'")

    waveform = preprocess_audio(audio_path)
    audio_duration = waveform.shape[-1] / TARGET_SAMPLE_RATE
    model_input = waveform.to(DEVICE)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)
        torch.cuda.synchronize(DEVICE)

    started = time.perf_counter()
    try:
        with torch.inference_mode():
            transcript = model(model_input, language, decoder)
    except RuntimeError as error:
        if "out of memory" not in str(error).lower():
            raise
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        raise RuntimeError(
            "IndicConformer ran out of GPU memory. Close other GPU workloads, "
            "use a shorter audio clip, or set DEVICE to CPU and rerun model loading."
        ) from error

    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    inference_time = time.perf_counter() - started
    allocated_gb, peak_gb = _gpu_memory_gb()

    return {
        "transcript": str(transcript).strip(),
        "language": language,
        "decoder": decoder,
        "audio_duration": audio_duration,
        "inference_time": inference_time,
        "real_time_factor": inference_time / audio_duration if audio_duration else float("inf"),
        "gpu_memory_allocated": allocated_gb,
        "gpu_memory_peak": peak_gb,
    }


print("Audio preprocessing and benchmark helpers are ready.")

In [ ]:
benchmark_audio = WHATSAPP_AUDIO

if not benchmark_audio.exists():
    raise FileNotFoundError(f"Benchmark audio file not found: {benchmark_audio}")

print("Running single-file CTC test...")
ctc_result = _run_inference(benchmark_audio, language="te", decoder="ctc")

print("\n" + "=" * 60)
print("INDICCONFORMER ASR BENCHMARK")
print("=" * 60)
print("Language:", ctc_result["language"])
print("Decoder:", ctc_result["decoder"])
print(f"Audio duration: {ctc_result['audio_duration']:.3f} s")
print(f"Inference time: {ctc_result['inference_time']:.3f} s")
print(f"Real-time factor: {ctc_result['real_time_factor']:.3f}")
print(f"GPU memory: {ctc_result['gpu_memory_allocated']:.3f} GB allocated; {ctc_result['gpu_memory_peak']:.3f} GB peak")
print("Transcript:", ctc_result["transcript"])
print("=" * 60)

In [ ]:
print("Running single-file RNNT test on the same audio...")
rnnt_result = _run_inference(benchmark_audio, language="te", decoder="rnnt")

print("\n" + "=" * 60)
print("INDICCONFORMER ASR BENCHMARK")
print("=" * 60)
print("Language:", rnnt_result["language"])
print("Decoder:", rnnt_result["decoder"])
print(f"Audio duration: {rnnt_result['audio_duration']:.3f} s")
print(f"Inference time: {rnnt_result['inference_time']:.3f} s")
print(f"Real-time factor: {rnnt_result['real_time_factor']:.3f}")
print(f"GPU memory: {rnnt_result['gpu_memory_allocated']:.3f} GB allocated; {rnnt_result['gpu_memory_peak']:.3f} GB peak")
print("Transcript:", rnnt_result["transcript"])
print("=" * 60)

In [ ]:
telugu_audio = benchmark_audio
print("Running Telugu CTC and RNNT checks...")
telugu_ctc_result = _run_inference(telugu_audio, language="te", decoder="ctc")
telugu_rnnt_result = _run_inference(telugu_audio, language="te", decoder="rnnt")

print("Telugu CTC transcript:", telugu_ctc_result["transcript"])
print("Telugu RNNT transcript:", telugu_rnnt_result["transcript"])
print("Telugu tests complete.")

In [ ]:
hindi_audio = benchmark_audio
print("Running Hindi CTC and RNNT checks...")
hindi_ctc_result = _run_inference(hindi_audio, language="hi", decoder="ctc")
hindi_rnnt_result = _run_inference(hindi_audio, language="hi", decoder="rnnt")

print("Hindi CTC transcript:", hindi_ctc_result["transcript"])
print("Hindi RNNT transcript:", hindi_rnnt_result["transcript"])
print("Hindi tests complete.")

In [ ]:
benchmark_results = [
    ctc_result,
    rnnt_result,
    telugu_ctc_result,
    telugu_rnnt_result,
    hindi_ctc_result,
    hindi_rnnt_result,
]

local_mic_audio = next(
    (candidate for candidate in MICROPHONE_WAV_CANDIDATES if candidate.exists()),
    None,
)
if local_mic_audio is not None:
    print("Running optional local microphone WAV benchmark:", local_mic_audio.name)
    mic_result = _run_inference(local_mic_audio, language="te", decoder="ctc")
    benchmark_results.append(mic_result)
    print("Microphone transcript:", mic_result["transcript"])
else:
    print("No local microphone WAV found; optional microphone benchmark skipped.")

print("\n" + "=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)
for result in benchmark_results:
    print(
        f"{result['language']:>2} | {result['decoder']:<4} | "
        f"duration={result['audio_duration']:.3f}s | "
        f"inference={result['inference_time']:.3f}s | "
        f"RTF={result['real_time_factor']:.3f} | "
        f"peak GPU={result['gpu_memory_peak']:.3f}GB"
    )

In [ ]:
def transcribe_audio(audio_path, language="auto", decoder="ctc"):
    """Transcribe one audio file and return production-oriented metrics."""
    return _run_inference(audio_path, language=language, decoder=decoder)


print("Reusable transcribe_audio(audio_path, language='auto', decoder='ctc') is ready.")
print("Note: language='auto' reports the model limitation because IndicConformer requires an explicit code.")

# Example:
# result = transcribe_audio(
#     WHATSAPP_AUDIO,
#     language="te",
#     decoder="ctc",
# )
# result